# Analyzing Flood Impact on Infrastructure

In [21]:
# Import libraries
import ee
import geemap
import pandas as pd
import geopandas as gpd

import os

## . Connect to GEE

In [22]:
# Authenticate GEE
ee.Authenticate()

#Initialize GEE
ee.Initialize()

### . Flood Parameters

In [23]:
# Pre-flood date
before_start = "2022-08-01"
before_end   = "2022-09-10"

# Flood date
after_start = "2022-09-14"
after_end   = "2022-11-18"

# Cloud percentage value
cloud_perc = 58

# Radius for speckle filtering (Sentinel-1)
speckle_radius = 50 # Unit is meter

flood_threshold = -15


### . Visualization Parameters

In [5]:
# Visualization Parameters for Boundary
vis_params_aoi = {'color': 'red', 'fillColor': '00000000'}

# Visualization Parameters for Sentinel-2 (True Colour - RGB)
vis_params_s2_rgb = {"min": 0.1, "max": 0.6, "bands": ["B4", "B3", "B2"], "gamma": 0.9}

# Visualization Parameters for Sentinel-2 (False colour)
vis_params_s2_fcc = {"min": 0.01, "max": 0.6, "bands": ["B8", "B4", "B3"], "gamma": 0.9}

# Visualization Parameters for Sentinel-1
vis_params_s1_vv = ...

# Visualization Parameters for GSW Permanent Water
vis_params_perm_water = {'min': 0,'max': 1,'palette': ["cyan"]}

## . Helper Functions

## . Area of Interest (AOI)

In [6]:
# AOI (GeoJSON)
aoi_geojson = {
    "type": "Polygon",
    "coordinates": [
        [
            [6.853409, 7.676968],
            [6.685181, 7.676968],
            [6.685181, 7.890588],
            [6.853409, 7.890588],
            [6.853409, 7.676968]
        ]
    ]
}

# Convert to GEE Feature
aoi_feature = ee.Feature(aoi_geojson)


aoi_geometry = aoi_feature.geometry()

# Default location to show the map
map_center_coords = 7.8175, 6.7730


In [7]:
# Show AOI on the map
aoi_map = geemap.Map(center=(7.8175, 6.7730), zoom=10)
aoi_map.add_basemap("SATELLITE")
aoi_map.addLayer(aoi_feature, vis_params_aoi, "AOI Flood")
aoi_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## . Get Admin Boundaries

In [8]:
# Get Nigeria Admin Boundary from FAO
fao_bdry_l0 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0") # Adm0
fao_bdry_l1 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level1") # Adm1
fao_bdry_l2 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2") #Adm2

# Get the boundary of Nigeria from FAO GAUL
nga_bdry_l0 = fao_bdry_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # Main Nigerian bdry
nga_bdry_l1 = fao_bdry_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # States bdry
nga_bdry_l2 = fao_bdry_l2.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # LGAs bdry

lokoja_by_name = nga_bdry_l2.filter(ee.Filter.eq("ADM2_NAME", "Lokoja"))

### . Boundary Visualization

In [9]:

# Add Nigeria boundaries
aoi_map.addLayer(nga_bdry_l0, {"color": "black"}, "NGA Bdry L0")
aoi_map.addLayer(nga_bdry_l1, {"color": "blue"}, "NGA State")
aoi_map.addLayer(nga_bdry_l2, {"color": "green"}, "NGA LGA L2")

# Add Lokoja
aoi_map.addLayer(lokoja_by_name, {"color": "black"}, "Lokoja (Name)")

# Display map
aoi_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## . Download and Pre-Process Sentinel-2

In [10]:
# Function to download Sentinel-2
def download_s2(start_date: str, end_date : str, cloud_perc: int, extent : ee.Feature | ee.Geometry):
    """
    Download Sentinel-2 images filtered by date, cloud cover, and geographic extent.

    Args:
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        cloud_perc (int): Maximum allowable cloud pixel percentage.
        extent (ee.Feature|ee.Geometry): The region to filter the collection by.

    Returns:
        ee.ImageCollection: The filtered Sentinel-2 collection.
    """
    img_collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                    .filterDate(start_date, end_date) # Dates filter
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_perc)) # Cloud filter
                    .filterBounds(extent)) # Boundary filter   

    img_count =  img_count = img_collection.size().getInfo() # Number of image in the collection

    print(f"There are {img_count} images in the Sentinel-2 image collection\n")
    
    return img_collection



# Function to apply cloud masking to Sentinel-2
# See reference: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_CLOUD_PROBABILITY
def mask_clouds_s2(image: ee.Image, cloud_thresh: int):
    """
    Apply cloud masking using cloud probability
    Uses the S2_CLOUD_PROBABILITY dataset to mask out cloudy pixels.

    Args:
        image (ee.Image): Sentinel-2 image with cloud probability band
        cloud_thresh (int): Cloud probability threshold (0-100)

    Returns:
        ee.Image: The cloud-masked image.
    """
    clear_threshold = cloud_thresh
    clouds = ee.Image(
            image.get("cloud_mask")
            ).select("probability")
    
    no_clouds = clouds.lt(clear_threshold)
    masked_image = image.updateMask(no_clouds).copyProperties(image, ["system:time_start"])
    
    return masked_image
    

# Masks from the 20m and 60m bands for bad data at scene edges
def mask_edges_s2(image : ee.Image):
    """
    Mask bad data at scene edges using 20m and 60m band masks.
    
    Args:
        image: ee.Image - Sentinel-2 image
    
    Returns:
        ee.Image - Image with edges masked
    """
    masked_image = image.updateMask(
                    image.select("B8A").mask().updateMask(image.select("B9").mask())
                    ).copyProperties(image, ["system:time_start"])
    
    return masked_image 


# Function to convert Sentinel-2 DN to reflectance
def scale_bands_s2(image: ee.Image):
        """
        Convert Sentinel-2 digital numbers to surface reflectance.
    
    Args:
        image: ee.Image - Sentinel-2 image with DN values
    
    Returns:
        ee.Image - Image with scaled reflectance values (0-1)
        """
        scaled_image = image.multiply(0.0001).copyProperties(image, ["system:time_start"]) # Multiply image by 0.0001 and copy "system:time_start" property 
        return scaled_image


# Function to resample bands in Sentinel-2 to 10m
def resample_bands_s2(image: ee.Image):
        """
        Resample 20m bands to 10m resolution using bilinear interpolation.
    
    Args:
        image: ee.Image - Sentinel-2 image with 10m and 20m bands
    
    Returns:
        ee.Image - Image with all bands at 10m resolution
        """
        bands_10m = image.select(["B2", "B3", "B4", "B8"]) # Select 10m bands
        bands_20m = image.select(["B5", "B6", "B7", "B8A", "B11", "B12"]) # Select 20m bands 
        resampled_image = bands_20m.resample("bilinear").reproject(
                                                            crs = bands_10m.projection(), 
                                                             scale=10)
        
        combined_10m = bands_10m.addBands(resampled_image) # Combine original 10m with 20m
        
        return combined_10m.copyProperties(image, ["system:time_start"])


# Function to compute spectral indices
def compute_indices_s2(image:ee.Image):
        """
        Compute spectral indices (NDVI, NDWI, MNDWI, EVI) for Sentinel-2 imagery.
    
    Args:
        image: ee.Image - Sentinel-2 image with required bands
    
    Returns:
        ee.Image - Original image with added spectral indices
        """
        
        ndvi = image.normalizedDifference(["B8", "B4"]).rename("ndvi")
        ndwi = image.normalizedDifference(["B3", "B8"]).rename("ndwi")
        mndwi = image.normalizedDifference(["B3", "B11"]).rename("mndwi")
        evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            "NIR":image.select("B8"),
            "RED": image.select("B4"),
            "BLUE": image.select("B2")
        }
        ).rename("evi")
        
        updated_image = image.addBands([ndvi, ndwi, mndwi, evi]).copyProperties(image, ["system:time_start"])
        
        return updated_image 



In [11]:
# Download Sentinel-2 images
s2_img_col = download_s2(after_start, after_end, cloud_perc, aoi_geometry)

# Apply cloud masking to ALL IMAGES in the Sentinel-2 collection
# Cloud mask is typically applied using the cloud probability dataset
s2_img_col = s2_img_col.map(lambda img: mask_edges_s2(img))  # Edge masking

# Resample bands for ALL IMAGES in the Sentinel-2 collection
s2_img_col = s2_img_col.map(lambda img: resample_bands_s2(img))

# Rescale pixel values  for ALL IMAGES in the Sentinel-2 collection
s2_img_col = s2_img_col.map(lambda img: scale_bands_s2(img))
                            
# Compute Indices for ALL IMAGES in the Sentinel-2 collection
s2_img_col = s2_img_col.map(lambda img: compute_indices_s2(img))

There are 15 images in the Sentinel-2 image collection



In [12]:
# Convert Sentinel-2 images to a List
s2_img_list = s2_img_col.toList(s2_img_col.size())

In [13]:
# Create a median composite for visualization
s2_composite = s2_img_col.median().clip(aoi_feature)

# Visualize Sentinel-2 imagery

s2_map = geemap.Map(center= map_center_coords, zoom=11)
s2_map.add_basemap("SATELLITE")

s2_map.addLayer(s2_composite, vis_params_s2_rgb, "Sentinel-2 RGB (Post-flood)")
s2_map.addLayer(s2_composite, vis_params_s2_fcc, "Sentinel-2 False Color (Post-flood)")
s2_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
s2_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## . Download and Pre-process Sentinel-1

In [14]:
# Function to download Sentinel-1
def download_s1(start_date: str, end_date: str, extent: ee.Feature | ee.Geometry):
    """
    """
    img_collection = (ee.ImageCollection("COPERNICUS/S1_GRD")
                        .filterDate(Afterstart_date, end_date) # Dates filter
                        .filter(ee.Filter.eq("instrumentMode", "IW")) # InstrumentMode
                        .filterMetadata("resolution_meters", "equals", 10) # Resolution filter
                        .filterBounds(aoi_feature.geometry()) # Boundary filter
                        .select(["VV", "VH"]))

    # Number of images in the Sentinel-1 Collection
    img_count = img_collection.size().getInfo()

    print(f"There are {img_count} Sentinel-1 Images")


# Function to apply Speckle filter to Sentinel-1 (focal median)
def focal_speckle_filter(image : ee.Image, radius : int,  unit: str = "meters"):
    """
    Focal Median filter with circle kernel.
    """
    image_filtered = image.focal_median(radius, "circle", unit)
    image_filtered.copyProperties(image, image.propertyNames())

    return image_filtered

In [32]:
# Download Sentinel-1 imagery
s1_img_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
                .filterDate(after_start, after_end))

# Check properties of Sentinel-1 Collection
s1_img_col.first().getInfo()



s1_img_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
                .filterDate(after_start, after_end)
                .filter(ee.Filter.eq("instrumentMode", "IW"))
                .filterMetadata("resolution_meters", "equals", 10)
                .filterBounds(aoi_geometry)
                .select(["VV", "VH"]))

# Dates of the images in the S-1 collection
s1_dates = s1_img_col.aggregate_array("system:time_start").getInfo()
s1_dates = [ee.Date(date).format("YYYY-MM-dd").getInfo() for date in s1_dates]
s1_dates

# Select S-1 image on "2022-10-13"
s1_oct_13 = s1_img_col.filterDate("2022-10-12", "2022-10-14").first()
s1_oct_13 = s1_oct_13.clip(aoi_feature)

s1_map = geemap.Map(center=map_center_coords, zoom=8)
s1_map.add_basemap("SATELLITE")

s1_map.addLayer(s1_oct_13, {"min": -20, "max": -5, "bands": ["VV"]}, "S1 2022-10-13")

# Apply Speckle filter
s1_oct_13_filtered = s1_oct_13.focal_median(50, "circle", "meters")

s1_map.addLayer(s1_oct_13_filtered , {"min": -20, "max": -5, "bands": ["VV"]}, "Filtered S1 2022-10-13")
s1_map


Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [33]:
# Convert Sentinel-1 images to a List
s1_img_list = s1_img_col.toList(s1_img_col.size())

In [36]:
# Image Histogram for Sentinel-1
import geemap.chart as chart

# Sample values as features from VV band
sampled_values_s1 = s1_oct_13_filtered.sample(region=aoi_feature.geometry(), 
                            scale=10, 
                            numPixels=10000)

# Options for the histogram plot
hist_options = {
        "title": "Image Distribution",
        "xlabel": "Values",
        "ylabel": "Pixel Count",
        "colors": ["#1d6b99"],
    }

# Distribution (backscatter) values using histogram 

s1_band_vv = "VV"

hist_s1 = chart.feature_histogram(
    features=sampled_values_s1,          
    property=s1_band_vv,         
    maxBuckets=50,                  
    **hist_options                    
)

print(hist_s1)


None


Extract Flood Mask Function

In [39]:
# From Sentinel-1
s1_threshold = -15

# Extract the flood mask
s1_flood_mask = (s1_oct_13_filtered.select(s1_band_vv)
                 .lt(s1_threshold)
                 .clip(aoi_feature.geometry())
                 .rename("flood_mask"))

# From Sentinel-1
s1_threshold = -15

# Extract the flood mask
s1_flood_mask = (s1_oct_13_filtered.select(s1_band_vv)
                 .lt(s1_threshold)
                 .clip(aoi_feature.geometry())
                 .rename("flood_mask"))

flood_map = geemap.Map(center=map_center_coords, zoom=11)
flood_map.add_basemap("SATELLITE")

flood_map.addLayer(s1_oct_13_filtered , {"min": -20, "max": -5, "bands": ["VV"]}, "Filtered S1 2022-10-13")
flood_map.addLayer(s1_flood_mask, {"min": 0, "max": 1, "bands" : ["flood_mask"], "palette": ["white", "blue"]}, "S1 - Flood Mask")
flood_map
flood_map.addLayer(s1_flood_mask, {"min": 0, "max": 1, "bands" : ["flood_mask"], "palette": ["white", "blue"]}, "S1 - Flood Mask")
flood_map


Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [ ]:
# Visualize Sentinel-1 imagery

In [ ]:
s1_img_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
                .filterDate(after_start, after_end)
                .filter(ee.Filter.eq("instrumentMode", "IW"))
                .filterMetadata("resolution_meters", "equals", 10)
                .filterBounds(aoi_geometry)
                .select(["VV", "VH"]))

# image.focal_median(radius, "circle", "meters")

## . Download Landsat-8

In [14]:
# Function to download Landsat-8
def download_l8(start_date: str, end_date: str, cloud_perc: int, extent: ee.Feature | ee.Geometry):
    """
    Download Landsat-8 Collection 2 Level 2 imagery filtered by date, cloud cover, and area.

    Args:
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        cloud_perc (int): Maximum allowed cloud cover percentage.
        extent (ee.Feature | ee.Geometry): Area of interest.

    Returns:
        ee.ImageCollection: Filtered Landsat-8 image collection.
    """
    img_collection = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUD_COVER", cloud_perc))
        .filterBounds(extent)
    )

    img_count = img_collection.size().getInfo()
    print(f"There are {img_count} images in the Landsat-8 image collection\n")

    return img_collection


# Band scaling factors for Landsat-8
def scale_factors_l8(image: ee.Image):
    """
    Apply scale factors to Landsat-8 optical and thermal bands.

    Args:
        image (ee.Image): Landsat-8 image.

    Returns:
        ee.Image: Scaled Landsat-8 image.
    """
    optical_bands = image.select(["SR_B.*"]).multiply(0.0000275).add(-0.2)
    thermal_bands = image.select(["ST_B.*"]).multiply(0.00341802).add(149.0)

    scaled_bands = image.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)

    return scaled_bands.copyProperties(image, ["system:time_start"])


# Cloud Masking for Landsat-8
def mask_l8_sr(image: ee.Image):
    """
    Mask clouds and cloud shadows in Landsat-8 SR imagery using the QA_PIXEL band.

    Args:
        image (ee.Image): Landsat-8 image.

    Returns:
        ee.Image: Cloud-masked image.
    """
    qa = image.select("QA_PIXEL")
    cloud = qa.bitwiseAnd(1 << 3).eq(0)
    cloud_shadow = qa.bitwiseAnd(1 << 4).eq(0)
    qa_mask = cloud.And(cloud_shadow)

    masked_image = image.updateMask(qa_mask).copyProperties(image, ["system:time_start"])

    return masked_image


# Function to compute spectral indices
def compute_indices_l8(image: ee.Image):
    """
    Compute spectral indices (NDVI, NDWI, MNDWI, EVI) for Landsat-8 imagery.

    Args:
        image (ee.Image): Landsat-8 image with required bands.

    Returns:
        ee.Image: Original image with added spectral index bands.
    """
    ndvi = image.normalizedDifference(["SR_B5", "SR_B4"]).rename("ndvi")
    ndwi = image.normalizedDifference(["SR_B3", "SR_B5"]).rename("ndwi")
    mndwi = image.normalizedDifference(["SR_B3", "SR_B6"]).rename("mndwi")

    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            "NIR": image.select("SR_B5"),
            "RED": image.select("SR_B4"),
            "BLUE": image.select("SR_B2")
        }
    ).rename("evi")

    updated_image = image.addBands([ndvi, ndwi, mndwi, evi]).copyProperties(
        image, ["system:time_start"]
    )

    return updated_image


In [15]:
# Convert Landsat-8 images to a List
l8_img_list = l8_img_col.toList(l8_img_col.size())

# Create a median composite for visualization
l8_composite = l8_img_col.median().clip(extent)

# Visualization parameters for Landsat-8 RGB
vis_params_l8_rgb = {
    "bands": ["SR_B4", "SR_B3", "SR_B2"],
    "min": 0,
    "max": 0.3
}

# Visualization parameters for false color composite
vis_params_l8_fcc = {
    "bands": ["SR_B5", "SR_B4", "SR_B3"],
    "min": 0,
    "max": 0.3
}

# Visualization parameters for AOI
vis_params_aoi = {
    "color": "red"
}


# Landsat-8 layers
aoi_map.addLayer(l8_composite, vis_params_l8_rgb, "Landsat-8 RGB (Post-flood)")
aoi_map.addLayer(l8_composite, vis_params_l8_fcc, "Landsat-8 False Color (Post-flood)")
aoi_map.addLayer(extent, vis_params_aoi, "AOI")
aoi_map

NameError: name 'l8_img_col' is not defined

## Download SRTM DEM

In [16]:
# Function to download SRTM DEM
def download_srtm(extent: ee.Geometry):
    """
    Download SRTM DEM for a given geographic extent.

    Args:
        extent (ee.Geometry): The region to clip the DEM to.

    Returns:
        ee.Image: The clipped SRTM DEM image.
    """
    dem = ee.Image("USGS/SRTMGL1_003").clip(extent)

    print("SRTM DEM loaded successfully.\n")

    return dem

In [17]:
srtm_dem = download_srtm(extent)

vis_params_dem = {
    "min": 0,
    "max": 3000,
    "palette": ["blue", "green", "yellow", "brown"]
}

aoi_map.addLayer(srtm_dem, vis_params_dem, "SRTM DEM")
aoi_map

NameError: name 'extent' is not defined

## Download Copernicus DEM GLO-30

In [18]:
# Function to download Copernicus DEM GLO-30
def download_copdem(extent: ee.Geometry):
    """
    Download Copernicus DEM GLO-30 for a given geographic extent.

    Args:
        extent (ee.Geometry): The region to clip the DEM to.

    Returns:
        ee.Image: The clipped Copernicus DEM image.
    """
    dem_collection = (ee.ImageCollection("COPERNICUS/DEM/GLO30")
                      .filterBounds(extent)
                      .select("DEM"))
    
    img_count = dem_collection.size().getInfo()
    print(f"There are {img_count} images in the Copernicus DEM collection\n")
    
    dem = dem_collection.mosaic().clip(extent)
    
    return dem

In [19]:
cop_dem = download_copdem(extent)

vis_params_dem = {
    "min": 0,
    "max": 3000,
    "palette": ["white", "yellow", "orange", "red"]
}

aoi_map.addLayer(cop_dem, vis_params_dem, "Copernicus DEM GLO-30")
aoi_map

NameError: name 'extent' is not defined

## . Download Building Dataset

In [ ]:
# Load Building Dataset (Google Open Buildings)
buildings = ee.FeatureCollection(
    "GOOGLE/Research/open-buildings-temporal/v1"
)

# Clip buildings to AOI
buildings_aoi = buildings.filterBounds(aoi_feature)

## . Flood Mapping

### . Histogram Analysis

In [20]:
import geemap.chart as chart

# Image Histogram for Sentinel-2

# Extract the NDWI band from Sentinel 2 composite
ndwi = s2_composite.select('ndwi')

# Sample values from the NDWI layer
sampled_values_s2 = ndwi.sample(
    region=aoi_geometry, 
    scale=10, 
    numPixels=10000
)

# Options for the histogram plot
hist_options = {
    "title": "NDWI Distribution - Sentinel-2",
    "xlabel": "NDWI Value",
    "ylabel": "Pixel Count",
    "colors": ["#1d6b99"],
}

# Generate Histogram
hist_s2 = chart.feature_histogram(
    features=sampled_values_s2,          
    property="ndwi",         
    maxBuckets=50,                  
    **hist_options                    
)
print(hist_s2)

None


### . Flood extraction using NDWI Thresholding approach

In [ ]:
def extract_floodmask_s2(image: ee.Image, s2_ndwi_threshold: float, extent: ee.Feature | ee.Geometry):
    """
    Extract flood mask from Sentinel-2 imagery using NDWI thresholding approach.

    
    Args:
        image (ee.Image): Sentinel-2 image
        s2_ndwi_threshold (float): NDWI threshold for water detection (default 0.0). Higher values detect more water.
        extent (ee.Feature|ee.Geometry): The region to clip by

    Returns:
        ee.Image: Binary flood mask (1 = flood, 0 = non-flood).
    """
    # Create the mask where NDWI is greater than the threshold
    mask = image.select("ndwi").gt(s2_ndwi_threshold).clip(extent).rename("flood_mask")
    
    # Return the mask, clipped to AOI
    return mask

In [29]:
# Apply the function to extract masks from all imagery Sentinel-2 collection
s2_flood_mask_col = s2_img_col.map(lambda img: extract_floodmask_s2(img, s2_ndwi_threshold=0.1, extent= aoi_geometry))

In [32]:
# Convert Sentinel-2 flood mask collection to a List for visualization
s2_flood_mask_list = s2_flood_mask_col.toList(s2_flood_mask_col.size())
flood_mask_count = s2_flood_mask_col.size().getInfo()

print(f"There are {flood_mask_count} flood masks in the collection")

There are 15 flood masks in the collection


In [34]:
# Visualize Sentinel 2 Flood mask
flood_map = geemap.Map(center=map_center_coords, zoom=11)
flood_map.add_basemap("SATELLITE")

flood_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
flood_map.addLayer(s2_composite, vis_params_s2_fcc, "Scaled Masked Image")

# Loop through all flood masks and add them to the map
for i in range(flood_mask_count):
    mask_img = ee.Image(s2_flood_mask_list.get(i))
    try:
        date = ee.Date(mask_img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    except:
        date = f"Mask_{i+1}"
    
    flood_map.addLayer(
        mask_img, 
        {"min": 0, "max": 1, "palette": ["white", "blue"]}, 
        f"Flood Mask - {date}"
    )

flood_map


Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## Flood Extraction from Landsat-8 using Thresholding approach

In [ ]:
def extract_floodmask_l8(image: ee.Image, l8_threshold: float):
    """
    Extract flood mask from Landsat-8 imagery using NDWI thresholding approach.

    Args:
        image (ee.Image): Landsat-8 image
        l8_threshold (float): NDWI threshold for water detection

    Returns:
        ee.Image: Binary flood mask (1 = flood, 0 = non-flood)
    """
    # Create flood mask where NDWI is greater than the threshold
    mask = image.select("ndwi").gt(l8_threshold).rename("flood_mask")

    # Return masked flood layer clipped to AOI
    return mask.updateMask(mask).clip(extent)

In [ ]:
# Apply it to all Landsat-8 images
l8_flood_mask = l8_img_col.map(lambda img: extract_floodmask_l8(img, l8_threshold=0.1))

# Create a composite flood mask for visualization
l8_flood_mask_med = l8_flood_mask.median()

# Add the flood mask to the already existing map
aoi_map.addLayer(
    l8_flood_mask_med,
    {"min": 0, "max": 1, "bands": ["flood_mask"], "palette": ["white", "blue"]},
    "Landsat-8 Flood Mask"
)

aoi_map

Map(bottom=125662.0, center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], p…

### . Download Global Surface Water (GSW) (Permanent water)

In [ ]:
def download_gsw(extent: ee.Feature | ee.Geometry, seasonality_threshold: int):
    """
    Download Global Surface Water (GSW) dataset and filter by seasonality.
    
    Args:
        extent (ee.Feature|ee.Geometry): The region to filter the collection by.
        seasonality_threshold (int): Number of months water presence (1-12). 

    Returns:
        ee.Image: The filtered GSW image with water presence layer.
    """
    # Load Global Surface Water dataset
    gsw = (ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
                  .select('seasonality')      # seasonality layer
                  .gte(seasonality_threshold) # Seasonality threshold ranges from 0-12 months
                  .clip(extent)               # Clipped to AOI
    )

    
    print(f"Permanent water loaded")
    
    return gsw


In [ ]:
# Download Global Surface Water data
perm_water = download_gsw(aoi_geometry, seasonality_threshold=10)

Permanent water loaded


In [ ]:
# Visualize Perm water
flood_map.addLayer(perm_water, vis_params_perm_water, "Permanet Water")
flood_map

Map(bottom=251023.0, center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], p…

## . Damage / Impact Assessment